Download own dataset from Roboflow through API

In [7]:
!pip install roboflow

from roboflow import Roboflow
rf = Roboflow(api_key="0kEFsvb8Ih23sdaqvBEs")
project = rf.workspace("c3423667-uon-edu-au").project("boboiboy-s-cnn-project")
version = project.version(3)
dataset = version.download("folder")


loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to BoBoiBoy's-CNN-Project-3 in folder:: 100%|██████████| 6227/6227 [00:00<00:00, 11843.75it/s]


Import required librareis for training

In [8]:
import os
import copy
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import datasets, transforms, models

from sklearn.metrics import classification_report, confusion_matrix

import random
import io
from PIL import Image
from torchvision import transforms

Model Configuration

In [9]:
# Dataset is availibale in Colab
# Expected layout:
# /contnent/...../
#   |____ train/<class_name>/*.jpg
#   |____ valid/<class_name>/*.jpg
#   |____ test/<class_name>/*.jpg
# Dataset root identifying
DATA_ROOT = "/content/BoBoiBoy's-CNN-Project-3"
print(f"Using dataset root: {DATA_ROOT}")


Using dataset root: /content/BoBoiBoy's-CNN-Project-3


Path, classes, training settings


In [18]:
# Turn dataset root into three folders the model uses
TRAIN_DIR = os.path.join(DATA_ROOT, "train")
VALID_DIR = os.path.join(DATA_ROOT, "valid")
TEST_DIR = os.path.join(DATA_ROOT, "test")

# Only accepts two class name
EXPECTED_CLASSES = ['apple','banana']

NUM_CLASSES = None
# Number of images processed in one step
BATCH_SIZE = 32
# Number of passess over the training set
NUM_EPOCHS = 5
# Larger learning rate value means bigger weight updates
LR = 1e-5
# Small penalty on large weights, help model generalise
WEIGHT_DECAY = 1e-4
# Locating the best weights and plots are saved.
BEST_CKPT = "best_model.pth"
CONFUSION_MATRIX_PATH = "confusion_matrix.png"
TRAINING_CURVES_PATH = "training_curves.png"
TEST_CONFUSION_MATRIX_PATH = "test_confusion_matrix.png"
# Scale pixel color into the range network expect
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
print(f"Dataset root: {DATA_ROOT}")

Device: cuda
Dataset root: /content/BoBoiBoy's-CNN-Project-3


Image Augmentation using torchvision.transforms

In [19]:
train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(15),
    transforms.RandomResizedCrop(224, scale = (0.8, 1.0)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

val_transform = transforms.Compose(
    [
        transforms.Resize(256),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    ]
)

class ImageFolderWithTrainMapping(Dataset):

  def __init__(self, root:str, classes:list, class_to_idx:dict, transform:None):
    self.root = root
    self.classes = classes
    self.class_to_idx = class_to_idx
    self.transform = transform
    self.samples = []
    exts =(".jpg", ".jpeg", ".png", ".webp")

    for class_name in classes:
      class_dir = os.path.join(root, class_name)
      if not os.path.isdir(class_dir):
        continue
      target = class_to_idx[class_name]
      for file_name in sorted(os.listdir(class_dir)):
        if file_name.lower().endswith(exts):
          self.samples.append((os.path.join(class_dir,file_name), target))

  def __len__(self):
    return len(self.samples)

  def __getitem__(self, index):
    path, target = self.samples[index]
    image = Image.open(path).convert("RGB")
    if self.transform is not None:
      image = self.transform(image)
    return image, target

train_dataset = datasets.ImageFolder(TRAIN_DIR, transform = train_transform)
classes = train_dataset.classes
class_to_idx = train_dataset.class_to_idx

NUM_CLASSES = len(classes)

missing_expected = sorted(set(EXPECTED_CLASSES) - set(classes))
unexpected_classes = sorted(set(classes) - set(EXPECTED_CLASSES))

if missing_expected:
  raise ValueError(f"Missing expected classes: {missing_expected}")
if unexpected_classes:
  raise ValueError(f"Unexpected classes: {unexpected_classes}")

valid_dataset = ImageFolderWithTrainMapping(VALID_DIR, classes, class_to_idx, transform = val_transform)
test_dataset = ImageFolderWithTrainMapping(TEST_DIR, classes, class_to_idx, transform = val_transform)

print(f"Train dataset size: {len(train_dataset)}")
print(f"Valid dataset size: {len(valid_dataset)}")
print(f"Test dataset size: {len(test_dataset)}")
print(f"Num of classes: {NUM_CLASSES}")
print(f"Classes: {classes}")

train_loader = DataLoader(train_dataset, batch_size = BATCH_SIZE, shuffle = True, num_workers = 2, pin_memory=(device.type == "cuda"),)
valid_loader = DataLoader(valid_dataset, batch_size = BATCH_SIZE, shuffle = False, num_workers = 2,pin_memory=(device.type == "cuda"),)
test_loader = DataLoader(test_dataset, batch_size = BATCH_SIZE, shuffle = False, num_workers = 2, pin_memory=(device.type == "cuda"),)

Train dataset size: 4980
Valid dataset size: 623
Test dataset size: 622
Num of classes: 2
Classes: ['apple', 'banana']


Transfer Learning on ResNet18

In [20]:
try:
    weights = models.ResNet18_Weights.IMAGENET1K_V1
    model = models.resnet18(weights=weights)
except Exception:
    model = models.resnet18(pretrained=True)

in_features = model.fc.in_features
model.fc = nn.Linear(in_features, len(classes))
for p in model.parameters():
    p.requires_grad = True
model = model.to(device)

Training Setup - Optimiser AdamW, Scheduler CosineAnnealingLR

In [21]:
train_targets = torch.tensor([t for _, t in train_dataset.samples], dtype = torch.long)
counts = torch.bincount(train_targets, minlength = len(classes)).float()
total_sample = counts.sum()
num_classes = len(classes)
class_weights = total_sample / (num_classes * counts.clamp(min=1.0))
class_weights = class_weights.to(device)

# Use the ResNet-18 model defined in the earlier cell instead of CustomFruitCNN
criterion = nn.CrossEntropyLoss(weight = class_weights)
optimizer = torch.optim.AdamW(model.parameters(), lr = LR, weight_decay = WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max = NUM_EPOCHS)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total params: {total_params:,} | Trainable: {trainable_params:,}")

Total params: 11,177,538 | Trainable: 11,177,538


Training loop and save the best checkpoint with val accuracy

In [22]:
def run_epoch(loader, train_mode:bool):
  if train_mode:
    model.train()
  else:
    model.eval()

  loss_sum = 0.0
  correct = 0
  total = 0

  for images, targets in loader:
    images = images.to(device)
    targets = targets.to(device)

    if train_mode:
      optimizer.zero_grad()
    with torch.set_grad_enabled(train_mode):
      logits = model(images)
      loss = criterion(logits, targets)

    if train_mode:
      loss.backward()
      optimizer.step()

    loss_sum += loss.item() * images.size(0)
    preds = logits.argmax(dim = 1)
    correct += (preds == targets).sum().item()
    total += targets.size(0)

  return loss_sum / max (total,1), correct / max(total,1)


Run traiing and keep the best weights

In [23]:
history = {
    "train_loss": [],
    "train_acc": [],
    "val_loss": [],
    "val_acc": []
}

best_val_acc = 0.0
best_state = None

for epoch in range(1, NUM_EPOCHS + 1):
  train_loss, train_acc = run_epoch(train_loader, train_mode = True)
  val_loss, val_acc = run_epoch(valid_loader, train_mode = False)
  scheduler.step()

  history["train_loss"].append(train_loss)
  history["train_acc"].append(train_acc)
  history["val_loss"].append(val_loss)
  history["val_acc"].append(val_acc)

  if val_acc > best_val_acc:
    best_val_acc = val_acc
    best_state = {
        "epoch": epoch,
        "model_state_dict": copy.deepcopy(model.state_dict()),
        "val_acc": best_val_acc,
        "classes": classes,
        "class_to_idx": class_to_idx,
    }
    torch.save(best_state, BEST_CKPT)

  print(
      f"Epoch {epoch:02d}/{NUM_EPOCHS} |"
      f"Train loss = {train_loss:.4f}, train accuracy={train_acc: .4f} |"
      f"Val loss = {val_loss:.4f}, val accuracy = {val_acc:.4f} |"
      f"Best val accuracy = {best_val_acc:.4f}"
  )
print(f"Training completed. Best val acc: {best_val_acc:.4f} --> saved to {BEST_CKPT}")


Epoch 01/5 |Train loss = 0.2366, train accuracy= 0.8936 |Val loss = 0.0312, val accuracy = 0.9984 |Best val accuracy = 0.9984
Epoch 02/5 |Train loss = 0.0306, train accuracy= 0.9964 |Val loss = 0.0117, val accuracy = 1.0000 |Best val accuracy = 1.0000
Epoch 03/5 |Train loss = 0.0132, train accuracy= 0.9986 |Val loss = 0.0080, val accuracy = 1.0000 |Best val accuracy = 1.0000
Epoch 04/5 |Train loss = 0.0102, train accuracy= 0.9986 |Val loss = 0.0067, val accuracy = 1.0000 |Best val accuracy = 1.0000
Epoch 05/5 |Train loss = 0.0116, train accuracy= 0.9972 |Val loss = 0.0075, val accuracy = 0.9968 |Best val accuracy = 1.0000
Training completed. Best val acc: 1.0000 --> saved to best_model.pth


Evaluation: best checkpoint, report, confusion matrix, training curves

In [26]:
try:
  checkpoint = torch.load(
      BEST_CKPT, map_location=device, weights_only=False
  )
except TypeError:
  checkpoint = torch.load(BEST_CKPT, map_location=device)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
  

Validation classification report

In [27]:
# Classification Report
def collect_preds(loader):
  all_preds = []
  all_targets = []
  with torch.no_grad():
    for images, targets in loader:
        images = images.to(device)
        logits = model(images)
        preds = logits.argmax(dim = 1).cpu().numpy()
        all_preds.append(preds)
        all_targets.append(targets.numpy())
  if not all_preds:
    return np.array([], dtype=int), np.array([], dtype=int)

  return np.concatenate(all_preds), np.concatenate(all_targets)


y_pred_valid, y_true_valid = collect_preds(valid_loader)
labels_list = list(range(len(classes)))

labels_list = list(range(len(classes)))
print("\n Classification report (validation): \n")
print(
    classification_report(
        y_true_valid,
        y_pred_valid,
        labels = labels_list,
        target_names = classes,
        zero_division = 0,
    )
)





 Classification report (validation): 

              precision    recall  f1-score   support

       apple       1.00      1.00      1.00       296
      banana       1.00      1.00      1.00       327

    accuracy                           1.00       623
   macro avg       1.00      1.00      1.00       623
weighted avg       1.00      1.00      1.00       623



Validation confusion matrix

In [28]:
from torch.serialization import save
# Confusion matrix
def plot_confusion_matrix(y_true, y_pred, labels_list, class_name, save_path, title):
  cm = confusion_matrix(y_true, y_pred, labels = labels_list)
  fig_cm, ax_cm = plt.subplots(figsize = (14,12))
  im_cm = ax_cm.imshow(cm, interpolation = "nearest", cmap = plt.cm.Blues)
  ax_cm.figure.colorbar(im_cm, ax = ax_cm)
  ax_cm.set(
      xticks = np.arange(cm.shape[1]),
      yticks = np.arange(cm.shape[0]),
      xticklabels = classes,
      yticklabels = classes,
      ylabel = "True label",
      xlabel = "Predicted label",
      title = "Confusion Matrix"
  )

  plt.setp(ax_cm.get_xticklabels(), rotation = 45, ha = "right", rotation_mode = "anchor")
  fig_cm.tight_layout()
  fig_cm.savefig(save_path, dpi = 300)
  plt.close(fig_cm)
  print(f"Saved confusion matrix to {save_path}")
  return cm

cm = plot_confusion_matrix(y_true_valid, y_pred_valid, labels_list, "Validation", CONFUSION_MATRIX_PATH, "Confusion Matrix")

Saved confusion matrix to confusion_matrix.png


Plot the training curves

In [29]:
# Training Curves
ephocs_axis = np.arange(1, len(history["train_loss"]) + 1)
fig_tr, (ax1, ax2) = plt.subplots(1, 2, figsize = (12,4))
ax1.plot(ephocs_axis, history["train_loss"], label = "Train")
ax1.plot(ephocs_axis, history["val_loss"], label = "Validation")
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Loss")
ax1.set_title("Training and Validation Loss")
ax1.legend()
ax1.grid(True, alpha = 0.3)

ax2.plot(ephocs_axis, history["train_acc"], label = "Train")
ax2.plot(ephocs_axis, history["val_acc"], label = "Validation")
ax2.set_xlabel("Epoch")
ax2.set_ylabel("Loss")
ax2.set_title("Training and Validation Accuracy")
ax2.legend()
ax2.grid(True, alpha = 0.3)

fig_tr.suptitle("Training curves")
fig_tr.tight_layout()
fig_tr.savefig(TRAINING_CURVES_PATH, dpi = 300)
plt.close(fig_tr)
print(f"Saved training curves to {TRAINING_CURVES_PATH}")

Saved training curves to training_curves.png


Final score on the test set

In [30]:
checkpoint = torch.load(BEST_CKPT, map_location = device)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

y_pred_test, y_true_test = collect_preds(test_loader)
labels_list = list(range(len(classes)))
test_acc = float((y_pred_test == y_true_test).sum()) / len(y_true_test)

print(f"Test sample: {len(y_true_test)}")
print(f"Test accuracy: {test_acc:.4f}")

print("\n Classification report (test): \n")
print(classification_report(y_true_test, y_pred_test, labels = labels_list, target_names = classes,zero_division = 0,))


plot_confusion_matrix(y_true_test, y_pred_test, labels_list, "Test", TEST_CONFUSION_MATRIX_PATH, "Confusion Matrix (Test)")

Test sample: 622
Test accuracy: 1.0000

 Classification report (test): 

              precision    recall  f1-score   support

       apple       1.00      1.00      1.00       330
      banana       1.00      1.00      1.00       292

    accuracy                           1.00       622
   macro avg       1.00      1.00      1.00       622
weighted avg       1.00      1.00      1.00       622

Saved confusion matrix to test_confusion_matrix.png


array([[330,   0],
       [  0, 292]])

In [ ]:
def collect_energies(loader):
  model.eval()
  enerries, lables = [], []
  with torch.no_grad():
    for images, targets in loader:
      images = images.to(device)
      logits = model(images)
      energy = -torch.logsumexp(logits, dim=1)
      enerries.append(energy.cpu())
      lables.append(targets)
    return torch.cat(enerries).numpy(), torch.cat(lables).numpy()

enerries, labels = collect_energies(test_loader)

energy_threshold = float(np.percentile(enerries, 95))
print(f"Energy threshold: {energy_threshold:.4f}")

for name, idx in class_to_idx.items():
  cls_energy = enerries[labels == idx]
  print(
      f"{name:6} n ={len(cls_energy):4d} "
      f"p95={np.percentile(cls_energy, 95):.4f} max={cls_energy.max():.4f}"
  )

Energy threshold: -1.3584
apple  n = 330 p95=-1.6188 max=-0.7468
banana n = 292 p95=-1.1530 max=-0.7271


Model Inference

In [35]:
import torch
import torch.nn as nn
from PIL import Image
from torchvision import transforms, models

# --- Load the trained model and setup for prediction ---
# Initialize ResNet18 model to match our saved checkpoint
try:
    weights = models.ResNet18_Weights.IMAGENET1K_V1
    model = models.resnet18(weights=weights)
except Exception:
    model = models.resnet18(pretrained=True)

# Model configuration and Initialisation
BEST_CKPT = "/content/best_model.pth"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load the best checkpoint to discover metadata
try:
    checkpoint = torch.load(BEST_CKPT, map_location=device, weights_only=False)
except TypeError:
    checkpoint = torch.load(BEST_CKPT, map_location=device)

classes = checkpoint["classes"]
class_to_idx = checkpoint["class_to_idx"]
num_classes = len(classes)

# Adjust the final fully connected layer of ResNet18
in_features = model.fc.in_features
model.fc = nn.Linear(in_features, num_classes)

# Load the state weights
model.load_state_dict(checkpoint["model_state_dict"])
model = model.to(device)
model.eval()

# Preprocessing transform
inference_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

# Prediction function
def predict_image(image_path: str, conf_threshold: float = 0.90, energy_threshold: float = -4.7):
  img = Image.open(image_path).convert("RGB")
  # Preprocess: adds batch dimension --> Shape: (1,3,224,224)
  tensor = inference_transform(img).unsqueeze(0).to(device)

  with torch.no_grad():
    logits = model(tensor)
    probs = torch.softmax(logits, dim=1)[0]
    energy = -torch.logsumexp(logits, dim=1).item()

    top_prob, pred_idx = torch.max(probs, dim=0)
    top_prob = top_prob.item()
    pred_idx = pred_idx.item()

  #if top_prob < conf_threshold or energy > energy_threshold:
  #  result_label = "Unknown / Neither Apple or Banana"
  #else:
  #  result_label = classes[pred_idx]
  result_label = classes[pred_idx]
  # Display results in console
  results = {
      "Status": result_label,
      "Top Probability": f"{top_prob * 100:.2f}% ",
      "Energy Score": f"{energy:.4f}",
      "Detailed Probabilities": {classes[i]: f"{probs[i].item() * 100:.2f}%" for i in range(num_classes)}
  }

  return results

Try one image with (softmax and energy)

In [36]:
# Banana
predict_image("/content/images (5).jpeg")

{'Status': 'apple',
 'Top Probability': '99.14% ',
 'Energy Score': '-2.9639',
 'Detailed Probabilities': {'apple': '99.14%', 'banana': '0.86%'}}

In [38]:
# Banana
predict_image("/content/images (7).jpeg")

{'Status': 'banana',
 'Top Probability': '99.97% ',
 'Energy Score': '-4.3751',
 'Detailed Probabilities': {'apple': '0.03%', 'banana': '99.97%'}}

In [41]:
# Bunch of Apple
predict_image("/content/bunch_of_apple.jpeg")

{'Status': 'apple',
 'Top Probability': '99.43% ',
 'Energy Score': '-2.7787',
 'Detailed Probabilities': {'apple': '99.43%', 'banana': '0.57%'}}

In [40]:
# Bunch of banana
predict_image("/content/images (8).jpeg")

{'Status': 'banana',
 'Top Probability': '99.09% ',
 'Energy Score': '-3.3022',
 'Detailed Probabilities': {'apple': '0.91%', 'banana': '99.09%'}}